In [82]:
import torch



A = torch.randn(size=(3, 10, 5))


In [96]:
U,S,V = torch.pca_lowrank(A, q=5, center=False, niter=2)

print('U shape: ', U.shape)
print('S shape: ', S.shape)
print('V shape: ', V.shape)

SV = torch.einsum('bq,bdq->bqd', S, V)
A_reconstructed = (U @ SV)
rec_loss = ((A - A_reconstructed)**2).sum()
print('Reconstruction loss: ', rec_loss)

U shape:  torch.Size([3, 10, 5])
S shape:  torch.Size([3, 5])
V shape:  torch.Size([3, 5, 5])
Reconstruction loss:  tensor(4.2487e-11)


In [97]:
U,S,V = torch.pca_lowrank(A, q=5, center=True, niter=2)

print('U shape: ', U.shape)
print('S shape: ', S.shape)
print('V shape: ', V.shape)

SV = torch.einsum('bq,bdq->bqd', S, V)
A_reconstructed = (U @ SV)
rec_loss = ((A - A_reconstructed)**2).sum()
print('Reconstruction loss: ', rec_loss)

U shape:  torch.Size([3, 10, 5])
S shape:  torch.Size([3, 5])
V shape:  torch.Size([3, 5, 5])
Reconstruction loss:  tensor(14.6313)


In [125]:
U,S,V = torch.pca_lowrank(A, q=5, center=True, niter=2)

print('U shape: ', U.shape)
print('S shape: ', S.shape)
print('V shape: ', V.shape)

SV = torch.einsum('bq,bdq->bqd', S, V)
A_reconstructed = (U @ SV)
rec_loss = ((A - A_reconstructed)**2).sum()
print('Reconstruction loss: ', rec_loss)

centered_data = A - A.mean(dim=1).unsqueeze(1)
rec_loss = ((centered_data - A_reconstructed)**2).sum()
print('Reconstruction loss of centered data: ', rec_loss)


centered_data = A - A.mean(dim=1).unsqueeze(1)
rec_loss = ((A - (A_reconstructed + A.mean(dim=1).unsqueeze(1)))**2).sum()
print('Reconstruction loss of centered data: ', rec_loss)

U shape:  torch.Size([10, 10, 5])
S shape:  torch.Size([10, 5])
V shape:  torch.Size([10, 5, 5])
Reconstruction loss:  tensor(50.0729)
Reconstruction loss of centered data:  tensor(8.5319e-11)
Reconstruction loss of centered data:  tensor(8.5707e-11)


tensor([[[-8.1350e-01,  2.5913e-01, -3.8862e-01,  1.1017e+00, -5.4058e-01],
         [-7.4788e-02,  2.3583e-02, -4.9840e-01, -7.1693e-01, -1.4195e-01],
         [-1.3847e-01,  6.4151e-01, -1.4545e+00, -3.8073e-02, -3.3887e-01],
         [ 1.5764e-01, -1.1450e+00,  1.9052e-01, -4.4236e-01, -4.0044e-01],
         [ 1.1372e+00,  1.1463e+00,  9.8498e-02, -4.1140e-01, -1.7831e+00],
         [ 2.2640e+00,  5.9864e-01, -2.6292e-01,  6.8343e-01,  5.5004e-01],
         [-3.4175e-01, -2.8841e-01, -8.7986e-01,  3.1358e-01,  7.5593e-01],
         [-2.9132e-01, -3.5650e-01,  5.2375e-01,  1.2243e+00,  1.6615e+00],
         [-1.0984e+00, -7.0058e-02,  9.4807e-01, -1.7699e+00,  3.4299e-04],
         [-8.0053e-01, -8.0927e-01,  1.7235e+00,  5.5673e-02,  2.3718e-01]],

        [[ 7.3737e-01,  1.1604e+00,  9.7257e-01,  4.8900e-01,  5.7001e-01],
         [ 1.5223e+00, -3.6285e-01, -1.9343e-01, -3.7826e-01, -3.1557e-01],
         [ 9.9414e-02, -1.2698e+00,  6.4678e-01,  2.0030e+00, -6.4187e-02],
         [

In [134]:
import torch

class PCAReconstructor:
    def __init__(self, q=5, niter=2):
        self.q = q
        self.niter = niter
        self.center = True
        

    def decompose(self, A):
        if self.center:
            mean = A.mean(dim=1, keepdim=True)
            
        U, S, V = torch.pca_lowrank(A, q=self.q, center=True, niter=self.niter)
        return U, S, V, mean

    def reconstruct(self, U, S, V, mean):    
        SV = torch.einsum('bq,bdq->bqd', S, V)
        A_reconstructed = U @ SV

        if self.center and mean is not None:
            A_reconstructed += mean
        return A_reconstructed

    def compute_reconstruction_loss(self, A):
        U, S, V, mean = self.decompose(A)
        A_reconstructed = self.reconstruct(U, S, V, mean)
       
        loss = ((A - A_reconstructed) ** 2).sum()
        return loss

# Example usage
A = torch.randn(10, 10, 5)  # Example tensor
pca_reconstructor = PCAReconstructor(q=5, niter=2)

# Perform decomposition
U, S, V, mean = pca_reconstructor.decompose(A)

# Reconstruct the matrix
A_reconstructed = pca_reconstructor.reconstruct(U, S, V, mean)

# Calculate reconstruction loss
loss = pca_reconstructor.compute_reconstruction_loss(A)
print('Reconstruction loss:', loss)


Reconstruction loss: tensor(7.1994e-11)


In [140]:
U.shape

torch.Size([10, 10, 5])